# Predviđanje cijena kuća — EDA i Predprocesiranje

**Završni projekt — Strojno i duboko učenje**

Dataset: [Kaggle — House Prices: Advanced Regression Techniques](https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
RANDOM_STATE = 42

## 1. Učitavanje podataka

Učitavamo train i test skupove te pregledavamo osnovne karakteristike: oblik, tipove podataka i prvih nekoliko redaka.

In [ ]:
train = pd.read_csv('../data/train.csv')
test  = pd.read_csv('../data/test.csv')

print('Train shape:', train.shape)
print('Test shape: ', test.shape)
train.head()

In [ ]:
train.dtypes.value_counts()

In [ ]:
train.describe()

## 2. Analiza ciljne varijable — SalePrice

Prikazujemo distribuciju sirove i logaritamski transformirane ciljne varijable. Log-transformacija (`np.log1p`) smanjuje asimetriju distribucije i poboljšava performanse regresijskih modela.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(train['SalePrice'], bins=50, edgecolor='black')
axes[0].set_title('SalePrice — sirova distribucija')
axes[0].set_xlabel('SalePrice')

axes[1].hist(np.log1p(train['SalePrice']), bins=50, edgecolor='black', color='steelblue')
axes[1].set_title('SalePrice — log1p distribucija')
axes[1].set_xlabel('log1p(SalePrice)')

plt.tight_layout()
plt.savefig('../output/figures/saleprice_dist.png', bbox_inches='tight')
plt.show()

## 3. Analiza nedostajućih vrijednosti

Identificiramo značajke s najvećim postotkom nedostajućih vrijednosti i vizualiziramo top 20.

In [ ]:
missing = train.isnull().mean().sort_values(ascending=False)
missing = missing[missing > 0]
print(f'Broj značajki s nedostajućim vrijednostima: {len(missing)}')
missing.head(20)

In [ ]:
top20_missing = missing.head(20)

plt.figure(figsize=(10, 6))
top20_missing.plot(kind='bar')
plt.title('Top 20 značajki prema % nedostajućih vrijednosti')
plt.xlabel('Značajka')
plt.ylabel('Udio nedostajućih vrijednosti')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('../output/figures/missing_values.png', bbox_inches='tight')
plt.show()

## 4. Korelacijska matrica

Prikazujemo heatmap top 10 numeričkih značajki s najvećom korelacijom s ciljnom varijablom `SalePrice`.

In [ ]:
numeric_cols = train.select_dtypes(include=[np.number]).columns
corr_with_target = train[numeric_cols].corr()['SalePrice'].abs().sort_values(ascending=False)
top10_features = corr_with_target.head(11).index.tolist()  # includes SalePrice itself

plt.figure(figsize=(10, 8))
corr_matrix = train[top10_features].corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', square=True)
plt.title('Korelacijska matrica — top 10 numeričkih značajki')
plt.tight_layout()
plt.savefig('../output/figures/correlation_heatmap.png', bbox_inches='tight')
plt.show()

## 5. Pipeline predprocesiranja

Spajamo train i test podatke kako bismo konzistentno primijenili sve transformacije:
1. Spajanje skupova
2. Popunjavanje nedostajućih vrijednosti (numeričke → medijan, kategoričke → mod)
3. Label encoding kategoričkih stupaca

In [ ]:
n_train = train.shape[0]

# 1. Concat
all_data = pd.concat([train.drop('SalePrice', axis=1), test], ignore_index=True)
print('all_data shape:', all_data.shape)

# 2. Fill missing
for col in all_data.columns:
    if all_data[col].dtype in [np.float64, np.int64]:
        all_data[col].fillna(all_data[col].median(), inplace=True)
    else:
        all_data[col].fillna(all_data[col].mode()[0], inplace=True)

# 3. Label encode all object columns
le = LabelEncoder()
for col in all_data.select_dtypes(include='object').columns:
    all_data[col] = le.fit_transform(all_data[col].astype(str))

print('Missing after preprocessing:', all_data.isnull().sum().sum())

## 6. Inženjering značajki

Dodajemo 4 nove značajke koje kombiniraju postojeće varijable i mogu poboljšati prediktivnu moć modela:
- `TotalSF` — ukupna kvadratura
- `TotalBath` — ukupan broj kupaonica
- `HouseAge` — starost kuće u trenutku prodaje
- `Remodeled` — je li kuća bila rekonstruirana

In [ ]:
all_data['TotalSF']   = all_data['TotalBsmtSF'] + all_data['1stFlrSF'] + all_data['2ndFlrSF']
all_data['TotalBath'] = all_data['FullBath'] + 0.5 * all_data['HalfBath']
all_data['HouseAge']  = all_data['YrSold'] - all_data['YearBuilt']
all_data['Remodeled'] = (all_data['YearRemodAdd'] != all_data['YearBuilt']).astype(int)

print('Nove značajke dodane. all_data shape:', all_data.shape)
all_data[['TotalSF', 'TotalBath', 'HouseAge', 'Remodeled']].describe()

## 7. Priprema skupova za treniranje

Razdvajamo nazad na train i test, primjenjujemo log-transformaciju na ciljnu varijablu, te dijelimo na train/validation split.

In [ ]:
# Split back
X = all_data[:n_train].reset_index(drop=True)
X_test_final = all_data[n_train:].reset_index(drop=True)

# Log-transform target
y = np.log1p(train['SalePrice'])

# Train/validation split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

print('X_train:', X_train.shape)
print('X_val:  ', X_val.shape)
print('X_test_final:', X_test_final.shape)

In [ ]:
# Save preprocessed data for use in notebook 2
import pickle

with open('../output/results/preprocessed_data.pkl', 'wb') as f:
    pickle.dump({
        'X_train': X_train,
        'X_val': X_val,
        'y_train': y_train,
        'y_val': y_val,
        'X_test_final': X_test_final
    }, f)

print('Podaci uspješno spremljeni.')